In [1]:
!git clone https://github.com/prava241/flash-attention.git
%cd flash-attention

Cloning into 'flash-attention'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 25 (delta 3), reused 24 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 6.53 KiB | 3.27 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/flash-attention


In [46]:
!git pull origin main

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
Unpacking objects: 100% (6/6), 1.48 KiB | 1.48 MiB/s, done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
From https://github.com/prava241/flash-attention
 * branch            main       -> FETCH_HEAD
   06891c0..a50299d  main       -> origin/main
Updating 06891c0..a50299d
Fast-forward
 notebooks/experiments.ipynb | 129 +++++++++++++++++++++++++++++++-------------
 src/main.cu                 |  19 +++++--
 2 files changed, 107 insertions(+), 41 deletions(-)


In [85]:
import torch
import numpy as np

N = 4096
D = 256

Q = torch.randn(N, D, device="cuda")
K = torch.randn(N, D, device="cuda")
V = torch.randn(N, D, device="cuda")

Q.cpu().numpy().astype(np.float32).tofile("data/q.bin")
K.cpu().numpy().astype(np.float32).tofile("data/k.bin")
V.cpu().numpy().astype(np.float32).tofile("data/v.bin")

In [19]:
!ls data

k.bin  output.bin  q.bin  v.bin


In [48]:
# Saves output to data/output.bin, reports runtime
# TODO: Profiles for bottlenecks
!nvcc -O3 src/main.cu src/kernels.cu -o attention_test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [97]:
!./attention_test 4096 256

baseline_attention runtime: 161.716 ms


In [106]:
import math

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()

# Naive PyTorch
S = Q @ K.T
S = S / math.sqrt(D)
P = torch.softmax(S, dim=-1)
O = P @ V

#Optimized PyTorch
# O = torch.nn.functional.scaled_dot_product_attention(Q, K, V)

end.record()
torch.cuda.synchronize()
elapsed_ms = start.elapsed_time(end)

print(f"pytorch_attention runtime: {elapsed_ms:.3f} ms")

reference = O.cpu().numpy()
output = np.fromfile("data/output.bin", dtype=np.float32).reshape(N, D)

print("Max abs error:", np.max(np.abs(reference - output)))
print("Mean abs error:", np.mean(np.abs(reference - output)))
print("All close:", np.allclose(reference, output, atol=1e-5, rtol=1e-5))

pytorch_attention runtime: 8.075 ms
Max abs error: 4.172325e-07
Mean abs error: 2.1177986e-08
All close: True
